In [15]:
import sys
!{sys.executable} -m pip install pandas numpy matplotlib scipy yfinance pandas_datareader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy.optimize import brentq
import yfinance as yf
from datetime import datetime

ERROR: Could not install packages due to an OSError: HTTPSConnectionPool(host='files.pythonhosted.org', port=443): Max retries exceeded with url: /packages/61/a9/16ea9346e1fc4a96e2896242d9bc674764fb9049b0044c0132502f7a771e/pandas-3.0.2-cp312-cp312-manylinux_2_24_aarch64.manylinux_2_28_aarch64.whl.metadata (Caused by NewConnectionError('<pip._vendor.urllib3.connection.HTTPSConnection object at 0xfcbd7346fa70>: Failed to establish a new connection: [Errno 101] Network is unreachable'))



ModuleNotFoundError: No module named 'pandas'

In [ ]:
def bs_call(s, k, t, r, sigma):
    d1=(np.log(s/k)+(r+0.5*sigma**2)*t)/(sigma*np.sqrt(t))
    d2=d1-sigma*np.sqrt(t)
    return s*norm.cdf(d1)-k*np.exp((-1)*r*t)*norm.cdf(d2)

def bs_putt(s, k, t, r, sigma):
    d1=(np.log(s/k)+(r+0.5*sigma**2)*t)/(sigma*np.sqrt(t))
    d2=d1-sigma*np.sqrt(t)
    return k*np.exp((-1)*r*t)*norm.cdf((-1)*d2)-s*norm.cdf((-1)*d1)

In [ ]:
def bs_call_greeks(s, k, t, r, sigma):
    d1 = (np.log(s/k) + (r + 0.5*sigma**2)*t) / (sigma*np.sqrt(t))
    d2 = d1 - sigma*np.sqrt(t)
    phi = norm.pdf(d1)
    N1, N2 = norm.cdf(d1), norm.cdf(d2)
    
    price = s*N1 - k*np.exp(-r*t)*N2
    delta = N1
    gamma = phi / (s*sigma*np.sqrt(t))
    vega  = s*phi*np.sqrt(s)
    theta = -s*phi*sigma/(2*np.sqrt(t)) - r*k*np.exp(-r*t)*N2
    rho   = k*t*np.exp(-r*t)*N2
    
    return {
        'price': price, 'delta': delta, 'gamma': gamma,
        'vega': vega, 'theta': theta, 'rho': rho,
        'vega_per_1pct': vega/100,
        'theta_per_day': theta/365,
        'rho_per_1pct':  rho/100,
    }

g = bs_call_greeks(100, 100, 1.0, 0.05, 0.2)
for k, v in g.items():
    print(f"{k:>15}: {v:10.6f}")

In [ ]:
def finite_diff_check(s, k, t, r, sigma, h=1e-4):
    analytic = bs_call_greeks(s, k, t, r, sigma)
    
    delta_fd = (bs_call(s+h, k, t, r, sigma) - bs_call(s-h, k, t, r, sigma)) / (2*h)
    gamma_fd = (bs_call(s+h, k, t, r, sigma) - 2*bs_call(s, k, t, r, sigma) 
                + bs_call(s-h, k, t, r, sigma)) / h**2
    vega_fd  = (bs_call(s, k, t, r, sigma+h) - bs_call(s, k, t, r, sigma-h)) / (2*h)
    theta_fd = (bs_call(s, k, t+h, r, sigma) - bs_call(s, k, t-h, r, sigma)) / (2*h)
    rho_fd   = (bs_call(s, k, t, r+h, sigma) - bs_call(s, k, t, r-h, sigma)) / (2*h)
    
    print(f"{'Greek':>8} {'Analytic':>12} {'Finite diff':>14} {'|diff|':>12}")
    for name, a, fd in [('delta', analytic['delta'], delta_fd),
                        ('gamma', analytic['gamma'], gamma_fd),
                        ('vega',  analytic['vega'],  vega_fd),
                        ('theta', analytic['theta'], theta_fd),
                        ('rho',   analytic['rho'],   rho_fd)]:
        print(f"{name:>8} {a:12.6f} {fd:14.6f} {abs(a-fd):12.2e}")

finite_diff_check(100, 100, 1.0, 0.05, 0.2)

In [ ]:
ticker="SPY"
tk=yf.Ticker(ticker)
hist=yf.download(ticker, period="1y", auto_adjust=True)["Close"]
s0=float(hist.iloc[-1].item())
print(f"Ціна{ticker}:${s0:.2f}")
logret=np.log(hist / hist.shift(1)).dropna()
sigma_hist=float(logret.std().item()*np.sqrt(252))
print(f"Історична σ:{sigma_hist*100:.2f}%")


In [ ]:
expiries=tk.options
print("Доступні експірації:", expiries[:10])
expiry=expiries[8]
t=(pd.Timestamp(expiry)-pd.Timestamp.today()).days/365
print(f"Обрана експірація:{expiry}, t={T:.4f} років")
chain=tk.option_chain(expiry)
calls=chain.calls.copy()
calls['mid']=(calls['bid']+calls['ask'])/2
calls=calls[calls['mid']>0]
calls[['strike', 'bid', 'ask', 'mid', 'volume', 'impliedVolatility']].head(10)

In [ ]:
try:
    from pandas_datareader import data as pdr
    r_series=pdr.DataReader("DGS3MO", "fred", start=pd.Timestamp.today()-pd.Timedelta(days=30))
    r=float(r_series.dropna().iloc[-1].item())/100
except Exception as e:
    print(f"FRED недоступний ({e}), використовуємо дефолт")
    r=0.045
print(f"r={r*100:.3f}%")